# 01 — Data Understanding

**AttriSense · Employee Attrition Prediction & Analytics System**

---

## Project Introduction

AttriSense predicts employee attrition and surfaces HR analytics from workforce data. Before any cleaning, feature engineering, or modeling, we need a clear picture of what the raw dataset contains.

This notebook performs **data understanding only**. No preprocessing, imputation, encoding, or row/column removal is applied here. Every figure and table is computed directly from the raw CSV.

**Dataset:** IBM HR Analytics Employee Attrition (`WA_Fn-UseC_-HR-Employee-Attrition.csv`)

**Goal of this notebook:** Document structure, quality, distributions, and column roles so later pipeline steps are informed by evidence rather than assumptions.

## Library Imports

We use Pandas for tabular inspection and Matplotlib for a single class-balance chart. The project package (`attrisense`) provides a shared loader so notebooks read from the path defined in `configs/config.yaml`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from attrisense.config import load_config
from attrisense.data import load_raw_data

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

config = load_config()
TARGET = config.target_column

## Dataset Loading

The raw file is loaded as-is. No transformations are applied at read time.

In [ ]:
df = load_raw_data()
df.head()

## Shape

Row and column counts establish dataset size and scope.

In [ ]:
n_rows, n_cols = df.shape
print(f"Rows   : {n_rows:,}")
print(f"Columns: {n_cols}")

## Data Types

Column dtypes determine which fields are numeric, categorical, or identifiers. This drives encoding and scaling decisions in later notebooks.

In [ ]:
dtype_overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null_count": df.notna().sum(),
    "unique_values": df.nunique(),
})
dtype_overview

## Missing Values

Missing data would require an imputation strategy. We count nulls per column across the full dataset.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_counts = missing[missing > 0]

print(f"Total missing values (all columns): {missing.sum()}")
print(f"Columns with missing values    : {len(missing_counts)}")

if missing_counts.empty:
    print("No missing values found.")
else:
    missing_counts

## Duplicates

Duplicate rows or repeated employee IDs would skew counts and leak information across splits. We check full-row duplicates and `EmployeeNumber` uniqueness separately.

In [ ]:
duplicate_rows = df.duplicated().sum()
duplicate_ids = df["EmployeeNumber"].duplicated().sum()

print(f"Duplicate rows          : {duplicate_rows}")
print(f"Duplicate EmployeeNumber: {duplicate_ids}")
print(f"Unique EmployeeNumber   : {df['EmployeeNumber'].nunique()}")

## Statistical Summary

Descriptive statistics for numeric columns: central tendency, spread, and observed min/max. Useful for spotting skew and scale differences before feature engineering.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
print(f"Numeric columns: {len(numeric_cols)}")

df[numeric_cols].describe().T

## Categorical Summary

For each non-numeric column we report the number of distinct values and the top category frequencies. This reveals cardinality and dominant classes.

In [ ]:
categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
print(f"Categorical columns: {len(categorical_cols)}\n")

for col in categorical_cols:
    print(f"--- {col} (unique: {df[col].nunique()}) ---")
    print(df[col].value_counts())
    print()

## Target Distribution

`Attrition` is the prediction target. Class balance affects metric selection and whether class weighting may be needed during modeling.

In [ ]:
target_counts = df[TARGET].value_counts()
target_pct = (df[TARGET].value_counts(normalize=True) * 100).round(2)

summary = pd.DataFrame({
    "count": target_counts,
    "percent": target_pct,
})
print(summary)
print()

fig, ax = plt.subplots(figsize=(6, 4))
colors = ["#3d9970", "#c0392b"]
bars = ax.bar(
    target_counts.index,
    target_counts.values,
    color=colors,
    edgecolor="white",
    linewidth=0.8,
)
ax.set_title("Attrition Target Distribution", fontsize=13, pad=12)
ax.set_xlabel("Attrition")
ax.set_ylabel("Employee Count")
ax.set_ylim(0, target_counts.max() * 1.15)
ax.spines[["top", "right"]].set_visible(False)

for bar, pct in zip(bars, target_pct.values):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 20,
        f"{pct:.1f}%",
        ha="center",
        va="bottom",
        fontsize=11,
    )

plt.tight_layout()
plt.show()

## Column Categorization

Grouping columns by role clarifies which fields are targets, identifiers, constants, and candidate features. Assignments are based on column names and the unique-value counts shown above.

In [ ]:
constant_cols = [col for col in df.columns if df[col].nunique() == 1]

column_groups = {
    "Target": [TARGET],
    "Identifier": ["EmployeeNumber"],
    "Constant (single value)": constant_cols,
    "Demographics": [
        "Age", "Gender", "MaritalStatus", "DistanceFromHome",
        "Education", "EducationField",
    ],
    "Job & Role": [
        "Department", "JobRole", "JobLevel", "JobInvolvement",
        "BusinessTravel", "OverTime",
    ],
    "Compensation": [
        "MonthlyIncome", "DailyRate", "HourlyRate", "MonthlyRate",
        "PercentSalaryHike", "StockOptionLevel",
    ],
    "Tenure": [
        "TotalWorkingYears", "YearsAtCompany", "YearsInCurrentRole",
        "YearsSinceLastPromotion", "YearsWithCurrManager",
        "NumCompaniesWorked",
    ],
    "Satisfaction & Performance": [
        "EnvironmentSatisfaction", "JobSatisfaction",
        "RelationshipSatisfaction", "WorkLifeBalance",
        "PerformanceRating",
    ],
    "Training": ["TrainingTimesLastYear"],
}

rows = []
for group, cols in column_groups.items():
    for col in cols:
        if col in df.columns:
            rows.append({
                "group": group,
                "column": col,
                "dtype": str(df[col].dtype),
                "unique_values": df[col].nunique(),
            })

column_map = pd.DataFrame(rows)
column_map

## Initial Observations

The points below are derived from the outputs in this notebook. Re-run all cells to refresh these numbers if the source file changes.

In [ ]:
# Compute observation metrics from the loaded dataset (no preprocessing).
obs_rows = df.shape[0]
obs_cols = df.shape[1]
obs_missing = int(df.isna().sum().sum())
obs_dup_rows = int(df.duplicated().sum())
obs_dup_ids = int(df["EmployeeNumber"].duplicated().sum())
obs_target_no = int((df[TARGET] == "No").sum())
obs_target_yes = int((df[TARGET] == "Yes").sum())
obs_target_yes_pct = round(obs_target_yes / obs_rows * 100, 2)
obs_constant = [c for c in df.columns if df[c].nunique() == 1]

print("Initial Observations")
print("=" * 50)
print(f"1. Dataset size: {obs_rows:,} employees, {obs_cols} columns.")
print(f"2. Missing values: {obs_missing} across all columns.")
print(f"3. Duplicate rows: {obs_dup_rows}; duplicate EmployeeNumber: {obs_dup_ids}.")
print(
    f"4. Target imbalance: {obs_target_yes} left ({obs_target_yes_pct}%), "
    f"{obs_target_no} stayed ({round(100 - obs_target_yes_pct, 2)}%)."
)
print(f"5. Constant columns (1 unique value): {obs_constant}.")
print(
    f"6. Numeric columns: {len(numeric_cols)}; "
    f"categorical columns: {len(categorical_cols)}."
)
print(
    "7. Satisfaction and rating fields (EnvironmentSatisfaction, "
    "JobSatisfaction, RelationshipSatisfaction, WorkLifeBalance) are stored "
    "as integers with 4 distinct levels each."
)
print(
    "8. JobRole has the highest categorical cardinality among non-constant "
    f"fields ({df['JobRole'].nunique()} roles)."
)